# Observability Production Engineering
### Structured Logging | Correlation IDs | RED Metrics | Distributed Tracing | ELK Stack

> **System:** ShopFlow -- 500k daily users, ELK + Prometheus + Jaeger stack.
> Format: **Mental Model -> Real Scenario -> BEFORE -> AFTER -> Frameworks -> Nuances**

*Shift+Enter to run each cell*

## Setup

In [ ]:
from __future__ import annotations
import time, json, uuid, contextvars, re
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Any
print('Setup OK')

---
## 1 · Structured Logging -- From Grep to Query

### Mental Model -- 'Lab Report vs Napkin Scribble'

```
WHAT   Log as machine-readable JSON fields, not freeform strings.
       Every piece of context is a named key, not embedded in a message.
WHY    Grep on freeform logs: fragile regex, no aggregation, no filtering.
       JSON logs: filter by ANY field, count errors per user, alert on thresholds.
HOW    Use a structured logger (structlog, python-json-logger).
       Always include: timestamp, level, event_name, request_id, service_name.
WHEN   Every application log statement -- no exceptions. Even debug logs.
```

### Nuance 1: Log EVENTS not STATES
BAD: `'Processing order'` -- tells you nothing useful.
GOOD: `'order.checkout.started', order_id=x, user_id=y, cart_total=z`
Events are queryable: 'How many orders started but never completed this hour?'

### Nuance 2: NEVER log secrets, PII, or card data
Logs are often stored less securely than the application itself. A developer
adding `log.debug('payment', card_number=card)` is a PCI DSS violation.
Use a redaction list: password, token, authorization, card, secret.

### Nuance 3: Log level discipline prevents log floods
DEBUG: turned off in prod. INFO: business events (order placed, user logged in).
WARNING: recoverable abnormalities (retry attempt, rate limit hit).
ERROR: exceptions that need investigation. CRITICAL: pager-worthy.
Log floods (DEBUG in prod) fill disks and obscure real errors.

### Real-World Scenario -- ShopFlow Payment Debugging

**Incident:** Stripe payments were failing silently for ~2% of EU customers.
The logs said: `'payment failed'` -- no user ID, no order ID, no error code.
Engineers spent 3 hours grepping logs trying to correlate failures.

**Fix:** Structured logging with `user_id`, `order_id`, `stripe_error_code`,
`region`, `card_country`. The failure pattern appeared in 10 minutes:
EU cards with 3DS challenges were timing out at 28s (before the 30s Stripe timeout).

In [ ]:
# Structured logging implementation

# Correlation ID lives in a contextvar -- set once per request, read everywhere
_request_id: contextvars.ContextVar[str] = contextvars.ContextVar('request_id', default='-')

_REDACT_KEYS = {'password', 'token', 'authorization', 'secret', 'card_number',
                'cvv', 'ssn', 'card', 'private_key'}

def _redact(fields: dict) -> dict:
    return {k: ('***REDACTED***' if k.lower() in _REDACT_KEYS else v)
            for k, v in fields.items()}


class StructuredLogger:
    def __init__(self, service: str):
        self.service = service
        self.records: list[dict] = []

    def _emit(self, level: str, event: str, **fields) -> dict:
        record = {
            'ts':         round(time.time(), 3),
            'level':      level,
            'service':    self.service,
            'event':      event,
            'request_id': _request_id.get(),
            **_redact(fields),
        }
        self.records.append(record)
        json.dumps(record)  # proves it is JSON-serializable
        return record

    def info(self, event, **f):  return self._emit('INFO', event, **f)
    def warn(self, event, **f):  return self._emit('WARN', event, **f)
    def error(self, event, **f): return self._emit('ERROR', event, **f)


log = StructuredLogger('payment-service')

# BEFORE -- freeform string logs
print('BEFORE (freeform logs):')
print('  payment failed for user 42')  # not queryable, no context
print('  Processing...\n')

# AFTER -- structured logs with context
token = _request_id.set(str(uuid.uuid4())[:8])

log.info('payment.started', user_id=42, order_id='ord-001',
         amount=99.99, currency='EUR', region='EU')

log.error('payment.failed',
          user_id=42, order_id='ord-001',
          stripe_error='card_error',
          stripe_code='authentication_required',
          duration_ms=28100,
          card='4242...4242',  # WILL BE REDACTED
          region='EU')

_request_id.reset(token)

print('AFTER (structured JSON logs):')
for r in log.records:
    print(json.dumps(r, indent=2))

print('\nCard number was redacted:', log.records[1]['card'])

### Where This Is Seen in Real Frameworks

| Tool | Structured logging |
|------|-------------------|
| **structlog** | `structlog.get_logger().info('event', key=val)` -- processors pipeline |
| **python-json-logger** | `logging.Formatter` that outputs JSON -- drop-in for stdlib logging |
| **Loguru** | `logger.bind(request_id=rid).info('event')` -- context binding |
| **FastAPI middleware** | Adds `request_id` to contextvar for every request |
| **ELK Stack** | Filebeat ships JSON logs -> Logstash -> Elasticsearch -> Kibana queries |

---
## 2 · Correlation IDs -- The Request DNA

### Mental Model -- 'The Package Tracking Number'

```
WHAT   A unique ID generated per request, carried in every log line and
       HTTP header as the request travels across services.
WHY    Without it: a failure in service C generates logs that can't be
       linked to the original user request in service A.
       With it: grep 'request_id=abc123' in Kibana shows the ENTIRE
       journey across 5 services in chronological order.
HOW    Generate UUID at the edge (API gateway or first service).
       Pass as X-Request-ID header. Store in contextvar. Log everywhere.
WHEN   ALWAYS. Every service should read and propagate this header.
```

### Nuance 1: Generate at the gateway, not at each service
If each service generates its own ID, you get 5 different IDs for one user request.
The gateway (Nginx, API gateway, first FastAPI middleware) generates ONE ID
and all downstream services inherit it via the header.

### Nuance 2: Accept and echo client-provided IDs (carefully)
If the client sends `X-Request-ID: client-generated-id`, echo it back and use it
in your logs. This lets the client correlate their logs with yours.
VALIDATE it: UUID format only. Never reflect arbitrary input (header injection risk).

### Nuance 3: Correlation ID != trace ID
Correlation ID ties logs together. Trace ID (OpenTelemetry) creates a span tree
with timing per service + per operation. Both are needed.
The correlation ID lets you grep; the trace ID lets you see WHERE the time went.

In [ ]:
import contextvars, uuid

_request_id: contextvars.ContextVar[str] = contextvars.ContextVar('request_id', default='-')
_UUID_RE = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$')

def start_request(client_id: str | None = None) -> str:
    # Accept client-provided ID only if it's a valid UUID (prevent injection)
    if client_id and _UUID_RE.match(client_id):
        req_id = client_id
    else:
        req_id = str(uuid.uuid4())
    _request_id.set(req_id)
    return req_id


def current_request_id() -> str: return _request_id.get()


# Simulate 3 services logging for the SAME user request
class MiniLogger:
    def __init__(self, name): self.name, self.lines = name, []
    def log(self, event, **kw):
        entry = {'svc': self.name, 'req': _request_id.get(), 'event': event, **kw}
        self.lines.append(entry)
        return entry


api_log     = MiniLogger('api-gateway')
order_log   = MiniLogger('order-service')
payment_log = MiniLogger('payment-service')

# Simulate one user request crossing 3 services
req_id = start_request('7c9e6679-7425-40de-944b-e07fc1f90ae7')  # client-provided

api_log.log('request.received', path='/checkout', method='POST')
order_log.log('order.creating', cart_items=3, total=49.99)
payment_log.log('payment.charging', amount=49.99, gateway='stripe')
payment_log.log('payment.succeeded', txn_id='pi_stripe_abc')
order_log.log('order.created', order_id='ord-007')
api_log.log('request.completed', status=201, duration_ms=245)

print(f'All log lines share request_id: {req_id}')
print()
all_lines = api_log.lines + order_log.lines + payment_log.lines
all_lines.sort(key=lambda x: x['svc'])  # in Kibana, sort by timestamp
for line in all_lines:
    print(f'  [{line["svc"]:20s}] {line["event"]:30s} req={line["req"][:8]}')

print('\nGrep request_id in Kibana -> see the ENTIRE journey across 3 services')

### Where This Is Seen in Real Frameworks

| Tool | Correlation ID |
|------|---------------|
| **FastAPI middleware** | `@app.middleware('http')` extracts/generates X-Request-ID, stores in contextvar |
| **OpenTelemetry** | W3C `traceparent` header propagates trace + span IDs |
| **Nginx** | `$request_id` variable; add `X-Request-ID: $request_id` to upstream headers |
| **AWS X-Ray** | `X-Amzn-Trace-Id` header -- AWS's correlation ID mechanism |
| **ELK / Kibana** | Filter `request_id: abc123` to see the full request timeline |

---
## 3 · RED Metrics -- What Matters in Production

### Mental Model -- 'The Car Dashboard'

```
WHAT   Three golden metrics per service/endpoint:
       R = Rate (requests per second)
       E = Error rate (% requests ending in 5xx)
       D = Duration (latency distribution: p50, p95, p99)
WHY    These three tell you everything about a service's health:
       - Rate drop = traffic loss (upstream issue or deploy problem)
       - Error spike = something is broken
       - p95 spike = users experiencing slow responses
HOW    Record (endpoint, duration_ms, ok/error) per request.
       Aggregate in Prometheus; visualize in Grafana.
WHEN   Every HTTP endpoint, every async worker, every batch job.
```

### Nuance 1: P95 matters more than mean
Mean latency: 99 requests at 10ms + 1 request at 1000ms = mean 19.8ms.
The 1 slow user is invisible in the mean. P95 = 1000ms reveals them.
For SLOs: define on P95 or P99, not mean.

### Nuance 2: Error rate vs error count
Error count spikes if traffic spikes. Error RATE (errors/total) is the real signal.
An alert on `error_count > 100` fires during every high-traffic period.
Alert on `error_rate > 1%` -- rate normalizes for traffic.

### Nuance 3: Latency histograms, not averages
Prometheus histograms are pre-aggregated into buckets (<10ms, <50ms, <100ms...)
Percentiles from histograms are approximate but cheap. Use `histogram_quantile(0.95, ...)`.

In [ ]:
# RED Metrics implementation
import random

@dataclass
class RedMetrics:
    _count:     dict = field(default_factory=lambda: defaultdict(int))
    _errors:    dict = field(default_factory=lambda: defaultdict(int))
    _durations: dict = field(default_factory=lambda: defaultdict(list))

    def record(self, endpoint: str, duration_ms: float, ok: bool) -> None:
        self._count[endpoint]    += 1
        if not ok: self._errors[endpoint] += 1
        self._durations[endpoint].append(duration_ms)

    def rate(self, endpoint: str) -> int:
        return self._count[endpoint]

    def error_rate(self, endpoint: str) -> float:
        n = self._count[endpoint]
        return (self._errors[endpoint] / n * 100) if n else 0

    def percentile(self, endpoint: str, p: float) -> float:
        xs = sorted(self._durations[endpoint])
        if not xs: return 0
        idx = int(len(xs) * p / 100)
        return xs[min(idx, len(xs)-1)]

    def report(self, endpoint: str) -> str:
        return (f'{endpoint}: '
                f'rate={self.rate(endpoint)} '
                f'errors={self.error_rate(endpoint):.1f}% '
                f'p50={self.percentile(endpoint, 50):.0f}ms '
                f'p95={self.percentile(endpoint, 95):.0f}ms '
                f'p99={self.percentile(endpoint, 99):.0f}ms')


metrics = RedMetrics()

# Simulate requests to two endpoints
random_state = random.Random(42)
for _ in range(200):
    dur = random_state.expovariate(1/50)  # mean 50ms
    metrics.record('/checkout', dur, ok=random_state.random() > 0.03)  # 3% error

for _ in range(1000):
    dur = random_state.expovariate(1/10)  # mean 10ms
    metrics.record('/products', dur, ok=True)

print('RED Metrics Report')
print('=' * 60)
print(metrics.report('/checkout'))
print(metrics.report('/products'))

print()
print('Key insight: mean vs p95 on /checkout:')
checkout_ms = metrics._durations['/checkout']
mean_ms = sum(checkout_ms) / len(checkout_ms)
p95_ms  = metrics.percentile('/checkout', 95)
print(f'  Mean: {mean_ms:.0f}ms  P95: {p95_ms:.0f}ms')
print('  P95 captures the slow tail; mean hides it')

### Where This Is Seen in Real Frameworks

| Tool | RED metrics |
|------|-------------|
| **prometheus_client** | `Counter`, `Histogram`, `Gauge` -- `histogram_quantile(0.95, ...)` |
| **Starlette/FastAPI** | `prometheus-fastapi-instrumentator` adds RED per-endpoint automatically |
| **Grafana** | Dashboard: rate=`rate(requests_total[5m])`, error%=`rate(errors[5m])/rate(total[5m])` |
| **Datadog** | `statsd` client: `statsd.timing('checkout.duration', ms)` |
| **AWS CloudWatch** | Percentile alarm: `p95 > 200ms` -> PagerDuty alert |

---
## 4 · Distributed Tracing -- Where Did the Time Go?

### Mental Model -- 'The Gantt Chart for a Request'

```
WHAT   A trace records the timeline of a request as a tree of spans.
       Each span = one operation (HTTP call, DB query, cache lookup).
       Spans nest: request span -> DB query span -> cache miss span.
WHY    Logs say WHAT happened. Metrics say HOW OFTEN. Traces say
       WHERE THE TIME WENT for a specific slow request.
       '200ms checkout' -> trace shows: 180ms waiting for Stripe API.
HOW    OpenTelemetry: instrument code with spans. Exporter sends to Jaeger/Tempo.
WHEN   Any request > 100ms, any distributed system, any latency regression.
```

### Nuance 1: Sampling is mandatory at scale
100% tracing at 10k req/s = 10k trace records/s = huge storage cost.
Head-based sampling: trace 1% randomly. Tail-based: trace 100% of slow/error requests.
Jaeger and Tempo support tail-based sampling.

### Nuance 2: Span attributes are searchable
Add `user_id`, `order_id`, `db.statement` as span attributes. In Jaeger,
search `order_id=ord-123` to find the exact trace for a user complaint.

### Nuance 3: Automatic instrumentation covers 80% of the work
OpenTelemetry auto-instruments FastAPI, SQLAlchemy, httpx, Redis.
You get DB query times, HTTP call times for free. Manual spans for
business logic (pricing calculation, fraud check).

In [ ]:
# Simulated distributed tracing (without OpenTelemetry dependency)

@dataclass
class Span:
    name:       str
    start_ms:   float
    duration_ms:float
    attributes: dict = field(default_factory=dict)
    children:   list = field(default_factory=list)

    def add_child(self, name: str, start_offset: float, dur: float, **attrs) -> 'Span':
        child = Span(name, self.start_ms + start_offset, dur, attrs)
        self.children.append(child)
        return child

    def render(self, indent: int = 0) -> list[str]:
        bar = '|' + '-' * int(self.duration_ms / 5)
        lines = [f'{" "*indent}{bar} {self.name} ({self.duration_ms:.0f}ms) '
                 + str(self.attributes)]
        for child in self.children:
            lines.extend(child.render(indent + 2))
        return lines


# Simulate a checkout request trace
root = Span('POST /checkout', 0, 245, {'user_id': 42, 'order_id': 'ord-007'})
root.add_child('auth.verify_token', 0, 3, **{'cache': 'hit'})
root.add_child('db.get_cart', 3, 8, **{'table': 'carts', 'rows': 3})
root.add_child('db.check_inventory', 11, 12, **{'table': 'inventory', 'skus': 3})
stripe_span = root.add_child('stripe.charge', 23, 198, **{'amount': 49.99})
stripe_span.add_child('http.POST api.stripe.com', 0, 185, **{'status': 200})
root.add_child('db.create_order', 221, 18, **{'table': 'orders'})
root.add_child('event.publish', 239, 6, **{'topic': 'order.placed'})

print('Checkout Request Trace (245ms total):')
print('Time ->  0                                    245ms')
print()
for line in root.render():
    print(line)

print()
print('Finding the bottleneck: Stripe call = 198ms / 245ms = 81% of request time')
print('Action: add timeout monitoring + circuit breaker on Stripe client')

### Where This Is Seen in Real Frameworks

| Tool | Tracing |
|------|----------|
| **OpenTelemetry** | `from opentelemetry import trace; tracer.start_as_current_span('name')` |
| **FastAPI** | `opentelemetry-instrumentation-fastapi` -- auto-instruments all routes |
| **SQLAlchemy** | `opentelemetry-instrumentation-sqlalchemy` -- DB queries as spans |
| **Jaeger** | Open-source trace collector + UI (`jaegertracing/all-in-one` Docker) |
| **Grafana Tempo** | Managed tracing; integrates with Loki (logs) via trace_id |